In [7]:
# ============================================================
# PREPARATION ML OPTIMISEE - PROJET IMMOBILIER DVF
# Base : travail de Carine, avec correction du risque de data leakage
# ============================================================

import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 0. PARAMETRES
# ------------------------------------------------------------

INPUT_FILE = "../../dvf_final_2020_2025.csv"

OUTPUT_CLEAN = "dvf_clean_model_ready_optimized.csv"
OUTPUT_X_TRAIN = "X_train_optimized.csv"
OUTPUT_X_TEST = "X_test_optimized.csv"
OUTPUT_Y_TRAIN = "y_train_optimized.csv"
OUTPUT_Y_TEST = "y_test_optimized.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("=" * 70)
print("PREPARATION ML OPTIMISEE - DVF 2020-2025")
print("=" * 70)

start_global = time.time()

# ------------------------------------------------------------
# 1. CHARGEMENT DU FICHIER FUSIONNE
# ------------------------------------------------------------

print("\n1. Chargement du fichier fusionne...")

usecols = [
    "id_mutation",
    "date_mutation",
    "annee",
    "mois",
    "nature_mutation",
    "valeur_fonciere",
    "surface_reelle_bati",
    "type_local",
    "nombre_pieces_principales",
    "surface_terrain",
    "code_departement",
    "code_commune",
    "code_postal",
    "adresse_numero",
    "adresse_nom_voie",
    "longitude",
    "latitude",
]

# On charge seulement les colonnes utiles pour economiser de la memoire
df = pd.read_csv(
    INPUT_FILE,
    usecols=lambda col: col in usecols,
    low_memory=False
)

print(f"Dataset charge : {df.shape[0]:,} lignes | {df.shape[1]} colonnes")
print("Colonnes chargees :")
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. PREMIER NETTOYAGE DE BASE
# ------------------------------------------------------------

print("\n2. Nettoyage de base...")

# On garde les transactions pertinentes pour le residentiel
df = df[df["nature_mutation"].isin(["Vente", "Vente en l'Ã©tat futur d'achÃ¨vement"])].copy()
df = df[df["type_local"].isin(["Maison", "Appartement"])].copy()

# Conversion des types
df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")
df["surface_terrain"] = df["surface_terrain"].fillna(0)
df["surface_reelle_bati"] = pd.to_numeric(df["surface_reelle_bati"], errors="coerce")
df["nombre_pieces_principales"] = pd.to_numeric(df["nombre_pieces_principales"], errors="coerce")
df["valeur_fonciere"] = pd.to_numeric(df["valeur_fonciere"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")

# Suppression des lignes sans informations essentielles
df = df.dropna(subset=[
    "id_mutation",
    "valeur_fonciere",
    "surface_reelle_bati",
    "longitude",
    "latitude",
    "code_commune",
    "code_departement"
])

print(f"Apres nettoyage de base : {df.shape[0]:,} lignes")

# ------------------------------------------------------------
# 3. REGROUPEMENT PAR TRANSACTION id_mutation
# ------------------------------------------------------------

print("\n3. Regroupement par id_mutation...")

# Une mutation peut etre presente sur plusieurs lignes.
# On regroupe pour obtenir une ligne = une transaction.
agg_rules = {
    "surface_reelle_bati": "sum",
    "nombre_pieces_principales": "sum",
    "surface_terrain": "sum",
    "valeur_fonciere": "max",
    "date_mutation": "first",
    "annee": "first",
    "mois": "first",
    "nature_mutation": "first",
    "type_local": "first",
    "code_departement": "first",
    "code_commune": "first",
    "code_postal": "first",
    "longitude": "first",
    "latitude": "first",
}

df_clean = df.groupby("id_mutation").agg(agg_rules).reset_index()

print(f"Apres regroupement : {df_clean.shape[0]:,} transactions")

# Recalcul du prix au m2 apres regroupement
df_clean["prix_m2"] = df_clean["valeur_fonciere"] / df_clean["surface_reelle_bati"]

# Suppression des valeurs impossibles
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=["prix_m2"])

print(f"Apres recalcul prix_m2 : {df_clean.shape[0]:,} lignes")

# ------------------------------------------------------------
# 4. NETTOYAGE GEOGRAPHIQUE
# ------------------------------------------------------------

print("\n4. Nettoyage geographique...")

before = len(df_clean)

df_clean = df_clean.dropna(subset=["longitude", "latitude"])
df_clean = df_clean[
    (df_clean["longitude"] != 0) &
    (df_clean["latitude"] != 0)
].copy()

after = len(df_clean)
print(f"Lignes supprimees pour coordonnees manquantes/aberrantes : {before - after:,}")

# ------------------------------------------------------------
# 5. TRAITEMENT DU CODE POSTAL
# ------------------------------------------------------------

print("\n5. Traitement du code postal...")

# Remplissage du code postal manquant par le code postal le plus frequent de la commune
if "code_postal" in df_clean.columns:
    df_clean["code_postal"] = df_clean.groupby("code_commune")["code_postal"].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )

    before = len(df_clean)
    df_clean = df_clean.dropna(subset=["code_postal"])
    after = len(df_clean)

    print(f"Lignes supprimees apres imputation code postal : {before - after:,}")

# ------------------------------------------------------------
# 6. NETTOYAGE DES SURFACES ET DES PIECES
# ------------------------------------------------------------

print("\n6. Nettoyage des surfaces et du nombre de pieces...")

# Un logement habitable a au moins une piece
df_clean.loc[df_clean["nombre_pieces_principales"] == 0, "nombre_pieces_principales"] = 1

# Calcul temporaire de la surface par piece
df_clean["surface_par_piece"] = (
    df_clean["surface_reelle_bati"] /
    df_clean["nombre_pieces_principales"].replace(0, 1)
)

before = len(df_clean)

# Filtrage des biens residentiels coherents
df_clean = df_clean[
    (df_clean["surface_reelle_bati"] >= 10) &
    (df_clean["surface_reelle_bati"] <= 500) &
    (df_clean["surface_par_piece"] >= 7) &
    (df_clean["surface_par_piece"] <= 80)
].copy()

after = len(df_clean)
print(f"Lignes supprimees pour surfaces incoherentes : {before - after:,}")

df_clean = df_clean.drop(columns=["surface_par_piece"])

# Conversion du nombre de pieces en entier
df_clean["nombre_pieces_principales"] = df_clean["nombre_pieces_principales"].astype(int)

# ------------------------------------------------------------
# 7. NETTOYAGE DU TERRAIN
# ------------------------------------------------------------

print("\n7. Nettoyage de la surface terrain...")

before = len(df_clean)

# Limite a 1 hectare pour rester sur du residentiel standard
df_clean = df_clean[df_clean["surface_terrain"] <= 10000].copy()

after = len(df_clean)
print(f"Lignes supprimees pour terrain > 10 000 m2 : {before - after:,}")

# ------------------------------------------------------------
# 7b. FILTRAGE DES PRIX EXTREMES
# ------------------------------------------------------------

print("\n7b. Filtrage des prix extremes...")

before = len(df_clean)
df_clean = df_clean[
    (df_clean["valeur_fonciere"] >= 20_000) &
    (df_clean["valeur_fonciere"] <= 2_000_000)
].copy()
after = len(df_clean)
print(f"Lignes supprimees pour prix aberrants (<20k ou >2M) : {before - after:,}")

# ------------------------------------------------------------
# ------------------------------------------------------------
# 7b. FILTRAGE DES PRIX EXTREMES
# ------------------------------------------------------------

print("\n7b. Filtrage des prix extremes...")

before = len(df_clean)
df_clean = df_clean[
    (df_clean["valeur_fonciere"] >= 20000) &
    (df_clean["valeur_fonciere"] <= 2000000)
].copy()
after = len(df_clean)
print(f"Lignes supprimees pour prix aberrants (<20k ou >2M) : {before - after:,}")

# 7c. FILTRAGE DES PRIX AU M2 ABERRANTS
before = len(df_clean)
df_clean = df_clean[(df_clean["prix_m2"] >= 300) & (df_clean["prix_m2"] <= 15_000)].copy()
after = len(df_clean)
print(f"Lignes supprimees pour prix_m2 aberrants (<300 ou >15000) : {before - after:,}")

# 8. FEATURE ENGINEERING SIMPLE
# ------------------------------------------------------------

print("\n8. Creation des variables explicatives...")

# Type de bien : maison ou appartement
df_clean["is_maison"] = (df_clean["type_local"] == "Maison").astype(int)

# Neuf / VEFA
df_clean["is_neuf"] = (
    df_clean["nature_mutation"] == "Vente en l'Ã©tat futur d'achÃ¨vement"
).astype(int)

# Anciennete de la transaction en mois
max_date_value = df_clean["annee"].max() * 12 + df_clean["mois"].max()
df_clean["anciennete_mois"] = max_date_value - (
    df_clean["annee"] * 12 + df_clean["mois"]
)

# Code departement : gestion de la Corse
df_clean["code_departement"] = df_clean["code_departement"].astype(str)
df_clean["code_departement"] = df_clean["code_departement"].replace({
    "2A": "201",
    "2B": "202"
})
df_clean["code_departement"] = pd.to_numeric(
    df_clean["code_departement"],
    errors="coerce"
)

df_clean = df_clean.dropna(subset=["code_departement"])
df_clean["code_departement"] = df_clean["code_departement"].astype(int)

df_clean["surface_par_piece"] = (df_clean["surface_reelle_bati"] / df_clean["nombre_pieces_principales"]).round(1)

print("Variables creees : is_maison, is_neuf, anciennete_mois, surface_par_piece")


# ------------------------------------------------------------
# 8b. ENRICHISSEMENT COMMUNAL (Filosofi, BPE, Population, Criminalite)
# ------------------------------------------------------------

print("\n8b. Enrichissement communal...")

ENRICH_PATH = "../../"

filosofi = pd.read_csv(f"{ENRICH_PATH}features_filosofi_clean.csv", dtype={"code_commune": str})
filosofi = filosofi[["code_commune", "revenu_median", "taux_pauvrete"]]

bpe = pd.read_csv(f"{ENRICH_PATH}features_bpe_clean.csv", dtype={"code_commune": str})
bpe = bpe[["code_commune", "nb_equipements_total"]]

pop = pd.read_csv(f"{ENRICH_PATH}features_population_clean.csv", dtype={"code_commune": str})
pop = pop[["code_commune", "population_2023", "evolution_pop_5_ans", "evolution_pop_10_ans"]]

crime = pd.read_csv(f"{ENRICH_PATH}features_criminalite_clean.csv", dtype={"code_commune": str})
crime = crime[["code_commune", "taux_cambriolages", "taux_vols_total", "taux_violences_total"]]

enrich = filosofi.merge(bpe, on="code_commune", how="outer")
enrich = enrich.merge(pop, on="code_commune", how="outer")
enrich = enrich.merge(crime, on="code_commune", how="outer")

ENRICH_COLS = [c for c in enrich.columns if c != "code_commune"]
for col in ENRICH_COLS:
    enrich[col] = enrich[col].fillna(enrich[col].median())

df_clean["code_commune"] = df_clean["code_commune"].astype(str)
df_clean = df_clean.merge(enrich, on="code_commune", how="left")
for col in ENRICH_COLS:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f"Enrichissement OK : {len(ENRICH_COLS)} nouvelles features ajoutees")
print(f"Shape df_clean apres enrichissement : {df_clean.shape}")

# ------------------------------------------------------------
# 9. SELECTION DES COLONNES AVANT SPLIT
# ------------------------------------------------------------

print("\n9. Selection des colonnes avant split...")

# On garde temporairement code_commune et prix_m2 pour creer les variables communales
# MAIS les variables communales seront calculees apres le split uniquement sur le train.
cols_model = [
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "surface_terrain",
    "valeur_fonciere",
    "annee",
    "mois",
    "code_departement",
    "longitude",
    "latitude",
    "is_maison",
    "is_neuf",
    "anciennete_mois",
    "surface_par_piece",
    # enrichissement communal
    "revenu_median",
    "taux_pauvrete",
    "nb_equipements_total",
    "population_2023",
    "evolution_pop_5_ans",
    "evolution_pop_10_ans",
    "taux_cambriolages",
    "taux_vols_total",
    "taux_violences_total",
    "code_commune",
    "prix_m2"
]

df_model = df_clean[cols_model].copy()

# Derniere securite : suppression des NaN restants
before = len(df_model)
df_model = df_model.dropna()
after = len(df_model)

print(f"Lignes supprimees pour NaN restants : {before - after:,}")
print(f"Dataset pret avant split : {df_model.shape[0]:,} lignes | {df_model.shape[1]} colonnes")

# ------------------------------------------------------------
# 10. SPLIT TRAIN / TEST AVANT VARIABLES COMMUNALES
# ------------------------------------------------------------

print("\n10. Split train/test AVANT calcul de commune_prix_m2...")

train_df, test_df = train_test_split(
    df_model,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Train : {train_df.shape[0]:,} lignes")
print(f"Test  : {test_df.shape[0]:,} lignes")

# ------------------------------------------------------------
# 11. CREATION DES VARIABLES COMMUNALES SANS DATA LEAKAGE
# ------------------------------------------------------------

print("\n11. Creation des variables communales (globales + par type) sans data leakage...")

# ── Statistiques globales (toutes transactions) ──────────────────────
stats_commune_train = train_df.groupby("code_commune").agg(
    commune_prix_m2=("prix_m2", "median"),
    commune_volume=("valeur_fonciere", "count")
).reset_index()

global_prix_m2_median = train_df["prix_m2"].median()
print(f"Communes dans le train : {stats_commune_train.shape[0]:,}")
print(f"Prix m2 median global train : {global_prix_m2_median:.2f}")

train_df = train_df.merge(stats_commune_train, on="code_commune", how="left")
test_df  = test_df.merge(stats_commune_train,  on="code_commune", how="left")
train_df["commune_prix_m2"] = train_df["commune_prix_m2"].fillna(global_prix_m2_median)
test_df["commune_prix_m2"]  = test_df["commune_prix_m2"].fillna(global_prix_m2_median)
train_df["commune_volume"]  = train_df["commune_volume"].fillna(0)
test_df["commune_volume"]   = test_df["commune_volume"].fillna(0)

# ── Dept median global + lissage bayesien global ─────────────────────
dept_mapping              = train_df.groupby("code_departement")["prix_m2"].median()
global_dept_prix_m2_median = train_df["prix_m2"].median()

train_df["dept_prix_m2"] = train_df["code_departement"].map(dept_mapping).fillna(global_dept_prix_m2_median)
test_df["dept_prix_m2"]  = test_df["code_departement"].map(dept_mapping).fillna(global_dept_prix_m2_median)

k = 15
train_df["commune_prix_m2"] = (train_df["commune_volume"] * train_df["commune_prix_m2"] + k * train_df["dept_prix_m2"]) / (train_df["commune_volume"] + k)
test_df["commune_prix_m2"]  = (test_df["commune_volume"]  * test_df["commune_prix_m2"]  + k * test_df["dept_prix_m2"])  / (test_df["commune_volume"]  + k)
print("commune_prix_m2 global (lissage bayesien k=15) OK.")

# ── Statistiques par type : MAISON et APPARTEMENT ───────────────────
for type_val, suffix in [(1, "maison"), (0, "appart")]:
    stats_type = (
        train_df[train_df["is_maison"] == type_val]
        .groupby("code_commune")
        .agg(**{
            f"commune_prix_m2_{suffix}": ("prix_m2", "median"),
            f"commune_volume_{suffix}": ("valeur_fonciere", "count")
        })
        .reset_index()
    )

    dept_type = (
        train_df[train_df["is_maison"] == type_val]
        .groupby("code_departement")["prix_m2"]
        .median()
    )
    global_dept_type = dept_type.median()

    train_df[f"dept_prix_m2_{suffix}"] = train_df["code_departement"].map(dept_type).fillna(global_dept_type)
    test_df[f"dept_prix_m2_{suffix}"]  = test_df["code_departement"].map(dept_type).fillna(global_dept_type)

    train_df = train_df.merge(stats_type, on="code_commune", how="left")
    test_df  = test_df.merge(stats_type,  on="code_commune", how="left")

    train_df[f"commune_prix_m2_{suffix}"] = train_df[f"commune_prix_m2_{suffix}"].fillna(train_df[f"dept_prix_m2_{suffix}"])
    test_df[f"commune_prix_m2_{suffix}"]  = test_df[f"commune_prix_m2_{suffix}"].fillna(test_df[f"dept_prix_m2_{suffix}"])
    train_df[f"commune_volume_{suffix}"] = train_df[f"commune_volume_{suffix}"].fillna(0)
    test_df[f"commune_volume_{suffix}"]  = test_df[f"commune_volume_{suffix}"].fillna(0)

    train_df[f"commune_prix_m2_{suffix}"] = (
        train_df[f"commune_volume_{suffix}"] * train_df[f"commune_prix_m2_{suffix}"] + k * train_df[f"dept_prix_m2_{suffix}"]
    ) / (train_df[f"commune_volume_{suffix}"] + k)
    test_df[f"commune_prix_m2_{suffix}"] = (
        test_df[f"commune_volume_{suffix}"] * test_df[f"commune_prix_m2_{suffix}"] + k * test_df[f"dept_prix_m2_{suffix}"]
    ) / (test_df[f"commune_volume_{suffix}"] + k)
    print(f"commune_prix_m2_{suffix} (lissage bayesien k=15) OK.")

# ── Baseline type-specifique (pour le residuel cible) ────────────────
train_df["commune_prix_m2_type"] = np.where(
    train_df["is_maison"] == 1,
    train_df["commune_prix_m2_maison"],
    train_df["commune_prix_m2_appart"]
)
test_df["commune_prix_m2_type"] = np.where(
    test_df["is_maison"] == 1,
    test_df["commune_prix_m2_maison"],
    test_df["commune_prix_m2_appart"]
)

# ── Feature d'interaction (base type-specifique) ─────────────────────
train_df["prix_estime_commune"] = train_df["commune_prix_m2_type"] * train_df["surface_reelle_bati"]
test_df["prix_estime_commune"]  = test_df["commune_prix_m2_type"]  * test_df["surface_reelle_bati"]
print("Feature prix_estime_commune (base type-specifique) ajoutee.")

# ── Residuel type-specifique (cible ML) ──────────────────────────────
train_df["residuel_prix_m2"] = train_df["prix_m2"] - train_df["commune_prix_m2_type"]
test_df["residuel_prix_m2"]  = test_df["prix_m2"]  - test_df["commune_prix_m2_type"]
print(f"Residuel type-specifique : min={train_df['residuel_prix_m2'].min():.0f}, max={train_df['residuel_prix_m2'].max():.0f}, mean={train_df['residuel_prix_m2'].mean():.0f}")

# ------------------------------------------------------------
# 12. CREATION X / y
# ------------------------------------------------------------

print("\n12. Creation des matrices X et y...")

target = "residuel_prix_m2"

drop_cols = [
    "valeur_fonciere",
    "code_commune",
    "prix_m2",
    "residuel_prix_m2",
    "commune_prix_m2_type"
]

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

y_train = train_df[target]
y_test = test_df[target]

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nColonnes finales utilisees pour le ML :")
print(X_train.columns.tolist())

print("\nVerification valeurs manquantes X_train :")
print(X_train.isna().sum())

print("\nVerification valeurs manquantes X_test :")
print(X_test.isna().sum())

# ------------------------------------------------------------
# 13. EXPORT DES FICHIERS
# ------------------------------------------------------------

print("\n13. Export des fichiers optimises...")

# Dataset complet optimise, utile pour audit
df_export = pd.concat([
    pd.concat([X_train, y_train], axis=1),
    pd.concat([X_test, y_test], axis=1)
], axis=0)

df_export.to_csv(OUTPUT_CLEAN, index=False)

X_train.to_csv(OUTPUT_X_TRAIN, index=False)
X_test.to_csv(OUTPUT_X_TEST, index=False)
y_train.to_csv(OUTPUT_Y_TRAIN, index=False)
y_test.to_csv(OUTPUT_Y_TEST, index=False)

print("Exports termines :")
print(f"- {OUTPUT_CLEAN}")
print(f"- {OUTPUT_X_TRAIN}")
print(f"- {OUTPUT_X_TEST}")
print(f"- {OUTPUT_Y_TRAIN}")
print(f"- {OUTPUT_Y_TEST}")

# ------------------------------------------------------------
# 14. RESUME FINAL
# ------------------------------------------------------------

total_time = time.time() - start_global

print("\n" + "=" * 70)
print("PREPARATION TERMINEE")
print("=" * 70)
print(f"Lignes train : {X_train.shape[0]:,}")
print(f"Lignes test  : {X_test.shape[0]:,}")
print(f"Nombre de variables explicatives : {X_train.shape[1]}")
print(f"Cible : {target}")
print(f"Temps total : {total_time/60:.2f} minutes")
print("=" * 70)

print("\nDifference importante avec la version initiale :")
print("- commune_prix_m2 est maintenant calculee uniquement sur le train.")
print("- Cela evite que des informations du test soient utilisees avant l'evaluation.")
print("- La methode est donc plus propre pour mesurer les performances reelles des modeles.")


PREPARATION ML OPTIMISEE - DVF 2020-2025

1. Chargement du fichier fusionne...
Dataset charge : 5,828,160 lignes | 17 colonnes
Colonnes chargees :
['id_mutation', 'date_mutation', 'annee', 'mois', 'nature_mutation', 'valeur_fonciere', 'surface_reelle_bati', 'type_local', 'nombre_pieces_principales', 'surface_terrain', 'code_departement', 'code_commune', 'code_postal', 'adresse_numero', 'adresse_nom_voie', 'longitude', 'latitude']

2. Nettoyage de base...
Apres nettoyage de base : 5,616,684 lignes

3. Regroupement par id_mutation...
Apres regroupement : 4,591,483 transactions
Apres recalcul prix_m2 : 4,591,483 lignes

4. Nettoyage geographique...
Lignes supprimees pour coordonnees manquantes/aberrantes : 0

5. Traitement du code postal...
Lignes supprimees apres imputation code postal : 0

6. Nettoyage des surfaces et du nombre de pieces...
Lignes supprimees pour surfaces incoherentes : 38,398

7. Nettoyage de la surface terrain...
Lignes supprimees pour terrain > 10 000 m2 : 16,813

7b